In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 2
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 2
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 2

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

/cloud-dev/.venv/lib/python3.13/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [ ]:
from transformers_sae.ops import generate

with torch.inference_mode():
    generate(
        ["The capital of France,", "The capital of the United Kingdom,"],
        model,
        tokenizer,
        stream=False,
    )



['<pad><pad><bos>The capital of France, Paris, is a city that is known for its rich history, culture, and architecture. It is', '<bos>The capital of the United Kingdom, London, is a city that is full of history and culture. It is also a city that is']


In [12]:
from transformers_sae.benchmark import BenchmarkModel

baseline_benchmark_model = BenchmarkModel(model, tokenizer)
print(
    # baseline_benchmark_model.generate("The capital of France,", None)
    baseline_benchmark_model.batch_generate(
        ["The capital of France,", "The capital of the United Kingdom,"], []
    )
)


[AnswerWrapper(answer='Paris, is a city that is known for its rich history, culture, and architecture. It is'), AnswerWrapper(answer='London, is a city that is full of history and culture. It is also a city that is')]


In [17]:
from transformers_sae.benchmark import MMLUBenchmark, MMLUTask

mmlu = MMLUBenchmark(1, tasks=[MMLUTask.HIGH_SCHOOL_BIOLOGY], )
baseline_results = mmlu.evaluate(model=baseline_benchmark_model)
# baseline_results = mmlu.evaluate(model=baseline_benchmark_model, batch_size=10)
print(baseline_results)

Processing high_school_biology:   0%|          | 0/1 [00:00<?, ?it/s]


MMLU Task Accuracy (task=high_school_biology): 1.0
Overall MMLU Accuracy: 1.0
overall_accuracy=1.0


In [23]:
mmlu.load_benchmark_dataset(MMLUTask.HIGH_SCHOOL_BIOLOGY)

[Golden(input='In a population of giraffes, an environmental change occurs that favors individuals that are tallest. As a result, more of the taller individuals are able to obtain nutrients and survive to pass along their genetic information. This is an example of\nA. directional selection.\nB. stabilizing selection.\nC. sexual selection.\nD. disruptive selection.\nAnswer:', actual_output=None, expected_output='A', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None)]

In [27]:
mmlu.predictions

,Task,Input,Prediction,Expected Output,Correct
0,high_school_biology,"In a population of giraffes, an environmental ...",A\n\nWhich of the following is not a character...,A,0
